In [ ]:
%load_ext autoreload
%autoreload 2
from utils import PATH_XE_AVERAGES
from GlobalFitter import GlobalFitter

import pandas as pd
import numpy as np
from pathlib import Path
from glob import glob
import os
import re

from waffles.data_classes.UniqueChannel import UniqueChannel
from waffles.np02_utils.AutoMap import expand_modules, dict_module_to_uniqch, getModuleName, getEndpointChannelFromModule
from monitoring.utils_monitoring import load_injections

import matplotlib.pyplot as plt
import dunestyle.matplotlib as dunestyle
from cycler import cycler
plt.rcParams.update( { "axes.prop_cycle":cycler(color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']) } )

In [ ]:
data_folder = Path("./data/")
analysis_name = 'average_waveforms_few_cuts_deconv_stable-hvieirad'
data_folder = data_folder / analysis_name

In [ ]:
df_inj = load_injections()
df_inj

In [ ]:
injppm = df_inj['ppm'].to_numpy()

In [ ]:
def genkeys(detectors:list):
    modules = expand_modules(detectors, dict_module_to_uniqch)
    keys = []
    for m in modules:
        if m[:2] in ["M1", "M2", "M3", "M4"]:#, "M6" ]:
            continue
        # if m in ["C3(2)", "C5(1)", "C5(2)", "C6(1)", "C6(2)"]:
        #     continue
        # if m in ["P05", "P06", "P15","P16", "P17", "P19", "P20", "P21", "P26", "P28", "P31"]:
        #     continue
        # if m not in ["P41", "P38", "P37", "P29", "P34", "P32", "P17", "P25", "P35"]:
        # if m not in ["P12", "P06", "P14", "P35"]:
        if m.startswith('P') and m not in ["P41", "P38", "P37", "P29", "P34", "P32", "P25", "P35", "P12", "P06", "P14"]:
            continue
        uch:UniqueChannel = dict_module_to_uniqch[m]
        ep, ch = uch.endpoint, uch.channel
        keys.append((ep,ch))

    return keys

In [ ]:
files = glob(f"{data_folder}/*.txt")
print(len(files), 'files found')
pattern = "ppm_([\d,\.]+)_ep_(\d+)_ch_(\d+).txt"
datasetfull = {}
results_per_ppm = {}
for file in files:
    fname = os.path.basename(file)
    match = re.match(pattern, fname)
    ppm = match.group(1)
    # if float(ppm) < 0.01:
    #     continue
    ep = int(match.group(2))
    ch = int(match.group(3))
    wf = np.loadtxt(file, dtype=np.float32)
    wf = wf - np.mean(wf[:40])
    datasetfull[ppm] = datasetfull.get(ppm, {})
    datasetfull[ppm][(ep, ch)] = wf

datasetfull = { ppm : v for ppm, v in sorted(datasetfull.items(), key=lambda x: float(x[0])) }


In [ ]:
trueppm = {}
for ppm in datasetfull.keys():
    if ppm == '0.00':
        trueppm[ppm] = 0.0
        continue
    imatch = np.argmin(abs(float(ppm) - injppm))
    trueppm[ppm]  = injppm[imatch]
trueppm

In [ ]:
modules = ["M", "C"]
if 'pmt' in analysis_name:
    modules = ["P"]
keys_fit = genkeys(modules)
print(keys_fit)
dataset_fit = { ppm: {k: w for k, w in sorted(kw.items(), key=lambda x: getModuleName(x[0][0], x[0][1])) if k in keys_fit} for ppm, kw in datasetfull.items() }
gFits = {}

In [ ]:
import time
for i, (ppm, dset) in enumerate(dataset_fit.items()):
    oneexp = True if ppm == '0.00' else False
    # if ppm != '0.00': continue
    # if float(ppm) < 5: continue
    # if float(ppm) > 2:
    #     dataset_fit[ppm].pop(getEndpointChannelFromModule("C6(1)"), {})
    #     dataset_fit[ppm].pop(getEndpointChannelFromModule("C6(2)"), {})
    error = 1 #if ppm != '0.00' else 5
    gFit = GlobalFitter(datasets=dset, offset_t0=320, penalty_strength=100, error=error)
    # gFit = GlobalFitter(datasets=dset, offset_t0=320, penalty_strength=1e5, error=200, prominence=500, penalty_scale=50)
    gFit.debug = True
    gFits[ppm] = gFit

    start = time.time()
    results = gFit.minimize(oneexp=oneexp,fit_limit_ns=12500, ppm=trueppm[ppm] )
    print(ppm, gFit.minuit.fmin.is_valid, gFit.minuit.fmin.reduced_chi2)
    # print(results)
    end = time.time()
    # print(end - start)
    results_per_ppm[ppm] = results
    


In [ ]:
def save_plot(gFit, keys, ppm, suffix, label, save=True):
    wfms = { key: wfm for key, wfm in gFit.datasets.items() if key in keys }
    keys = [ key for key in keys if key in gFit.datasets.keys() ] 
    fig, axs = gFit.plot_results(keys=keys, ncols=2, logscale='log' in suffix)
    for i, ax in enumerate(np.ravel(axs)):
        if i >= len(keys):
            continue
        if 'log' in suffix:
            ax.set_ylim(1e-3, 1.2*np.max(wfms[keys[i]]))
        else:
            ax.set_ylim(None, 1.2*np.max(wfms[keys[i]]))
            ax.set_xlim(40*16, 3000)
        # ax.set_xlim(55*16,120*16)
        # ax.set_ylim(-50,50)
    if save:
        plt.savefig(f'graphs_global/globalfit_{label}{suffix}_{ppm}.png')
    else:
        plt.show()
    plt.close()

for ppm, gFit in gFits.items():
    print(ppm)
    save=True
    # if ppm != '0.00': continue
    logscale=True
    suffix = "_log" if logscale else ""
    if "M" in modules:
        save_plot(gFit, genkeys(["M"]), ppm, suffix, "membrane", save=save)
    if "C" in modules:
        save_plot(gFit, genkeys(["C"]), ppm, suffix, "cathode", save=save)
    if "P" in modules:
        save_plot(gFit, genkeys(["P"]), ppm, suffix, "pmt", save=save)

In [ ]:
gFit = gFits['1.00']
x = np.array(gFit.values_penalty)
y = np.array(gFit.values_chi2)
print(len(x))
mask = np.ones_like(y, dtype=bool)
mask = y < 1e3
x = x[ mask ]
y = y[ mask ]
plt.plot(x, y, '.')
# plt.xscale('log')

# plt.xlim(1e1,1e2)
# plt.yscale('log')

In [ ]:
global_rows = []
module_rows = []

for ppm, results in results_per_ppm.items():
    global_rows.append({
        'ppm':    trueppm[ppm],
        'tta':    results['tta'].value*1e-3,
        'ttx':    results['ttx'].value*1e-3,
        'errtta': results['tta'].error*1e-3,
        'errttx': results['ttx'].error*1e-3,
    })
    for (ep, ch), r in results.modules.items():
        module_rows.append({
            'ppm':    trueppm[ppm],
            'ep':     ep,
            'ch':     ch,
            'module': getModuleName(ep, ch),
            'A':      r['A'].value,
            'B':      r['B'].value,
            'C':      r['C'].value,
            'errA':   r['A'].error,
            'errB':   r['B'].error,
            'errC':   r['C'].error,
            't0':     r['t0'].value,
            'sigma':  r['sigma'].value,
            # 'offset': r['offset'].value,
            'tta':    results['tta'].value*1e-3,
            'ttx':    results['ttx'].value*1e-3,
            'errtta': results['tta'].error*1e-3,
            'errttx': results['ttx'].error*1e-3,
        })

df  = pd.DataFrame(global_rows)

df.loc[df['ppm'] == 0, ['ttx']] = np.nan

dfm = pd.DataFrame(module_rows)

print(df)
print(dfm)


In [ ]:
plt.figure(figsize=(8,6))
plt.errorbar(df['ppm'], df['ttx'], yerr=df['errttx'], xerr=0.05, fmt='.', label=r'$\tau_{TX}$')
plt.errorbar(df['ppm'], df['tta'], yerr=df['errtta'], xerr=0.05, fmt='.', label=r'$\tau_{TA}$')
plt.ylabel(r'${\tau} [\mu s]$')
plt.xlabel(r'Concentration [ppm]')
plt.legend(loc='upper right')

plt.figure(figsize=(8,6))
plt.errorbar(df['ppm'], 1/df['ttx'], yerr=df['errttx']/df['ttx']**2, xerr=0.05, fmt='.', label=r'$\tau_{TX}$')
plt.errorbar(df['ppm'], 1/df['tta'], yerr=df['errtta']/df['tta']**2, xerr=0.05, fmt='.', label=r'$\tau_{TA}$')
plt.ylabel(r'$1\,/\,{\tau} [\mu s]$')
plt.xlabel(r'Concentration [ppm]')
plt.legend(loc='upper left')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.odr import ODR, Model, RealData
from dataclasses import dataclass
from typing import Optional, Callable
import pandas as pd


@dataclass
class ODRResult:
    slope:         float
    intercept:     float
    err_slope:     float
    err_intercept: float

    def __repr__(self):
        return (f"slope = {self.slope:.4f} ± {self.err_slope:.4f}, "
                f"intercept = {self.intercept:.4f} ± {self.err_intercept:.4f}")


def fit_inverse_tau(ppm, tau, err_tau, ppm_err=0.05, beta0=[1, 0]) -> ODRResult:
    """ODR linear fit on 1/tau vs ppm. Returns ODRResult."""
    data = RealData(ppm, 1/tau, sx=ppm_err, sy=err_tau/tau**2)
    out  = ODR(data, Model(lambda B, x: B[0]*x + B[1]), beta0=beta0).run()
    return ODRResult(
        slope         = out.beta[0],
        intercept     = out.beta[1],
        err_slope     = out.sd_beta[0],
        err_intercept = out.sd_beta[1],
    )


def plot_inverse_tau(ax, ppm, tau, err_tau, result, label, color, ppm_err=0.05, coeff=["A", "B"]):
    """Plot data points + ODR fit line on a given axis."""
    x_fit = np.linspace(ppm.min(), ppm.max(), 200)
    y_fit = result.slope * x_fit + result.intercept

    ax.errorbar(ppm, 1/tau, yerr=err_tau/tau**2, xerr=ppm_err,
                fmt='.', label=label, color=color)
    ax.plot(x_fit, y_fit, color=color,
            label=(rf'Fit {label}: '
                   rf'${result.intercept:.3f}\pm{result.err_intercept:.3f}$ + '
                   rf'${result.slope:.3f}\pm{result.err_slope:.3f} \cdot \text{{Xe [ppm]}}$'))


def run_tau_analysis(df: pd.DataFrame,
                     tau_col: str,
                     err_col: str,
                     row_filter: Optional[Callable] = None,
                     ppm_err: float = 0.05) -> tuple[ODRResult, pd.DataFrame]:
    """
    Filter df, fit 1/tau vs ppm, return (ODRResult, filtered_df).

    row_filter: any callable that takes a DataFrame and returns a filtered DataFrame, e.g.:
        row_filter = lambda df: df[df["ppm"] <= 4]
        row_filter = lambda df: df[df["ppm"].isin([1, 2, 3])]
        row_filter = lambda df: df[df["ppm"] >= 2][df["tta"] < 1200]
    """
    data = df.copy()
    if row_filter is not None:
        data = row_filter(data)

    result = fit_inverse_tau(data["ppm"], data[tau_col], data[err_col], ppm_err)
    return result, data


# --- Usage ---
result_tx, data_tx = run_tau_analysis(
    df, "ttx", "errttx",
    row_filter = lambda df: df[df["ppm"] > 0.1],
)
result_ta, data_ta = run_tau_analysis(
    df, "tta", "errtta",
    row_filter = lambda df: df[~df["ppm"].between(0.008, 0.02)],
)

fig, ax = plt.subplots(figsize=(8, 6))

plot_inverse_tau(ax, data_ta["ppm"], data_ta["tta"], data_ta["errtta"],
                 result_ta, label=r'$\tau_{TA}$', color='tab:orange')

plot_inverse_tau(ax, data_tx["ppm"], data_tx["ttx"], data_tx["errttx"],
                 result_tx, label=r'$\tau_{TX}$', color='tab:blue',
                 coeff=["C", "D"])

ax.set_ylabel(r'$1\,/\,\tau\;[\mu s^{-1}]$')
ax.set_xlabel(r'Xe [ppm]')
ax.legend(loc='upper left')
fig.tight_layout()
# plt.savefig('graphs_global/linear_global_pmt.png', dpi=300)


plt.show()
# print(f"TA: {result_ta}")
# print(f"TX: {result_tx}")

In [ ]:
from waffles.np02_utils.PlotUtils import get_style_module
def plot_modules(dfm, letter='C'):
    plt.figure(figsize=(12,6))
    for module, group in dfm.groupby("module"):
        if not module.startswith(letter):
            continue
        group = group.sort_values("ppm")
        group['total'] = group['A'] + group['B'] + group['C']
        group['slow'] = group['B'] + group['C']
        
        group = group[group["ppm"]>0.1]
        plt.plot(group["ppm"], group["C"]/(group['total']), label=module, marker='o', **get_style_module(module))
        # plt.plot(group["ppm"], group["A"]/(group['total']), label=module, marker='o', **get_style_module(module))
        # plt.plot(group["t0"], label=module, marker='o', **get_style_module(module))
    
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", ncols=2)
    plt.ylabel('Fraction C')
    plt.xlabel('Concentration [ppm]')
    
    plt.tight_layout()
    # plt.savefig(f'./graphs_global/fractions_{letter}.png')
plot_modules(dfm, 'M')
plot_modules(dfm, 'C')
# plot_modules(dfm, 'P')

    
